# Run Bradford Bulls Track-Level Annotation on Google Colab

Notebook này được thiết kế để chạy pipeline trên Google Colab. Nó sẽ tự động:
1. Clone repository từ Github về Colab (nhánh `feat/v3`)
2. Cài đặt thư viện và chuẩn bị weights YOLO
3. Chạy pipeline trích xuất keyframe và clip (chỉ yêu cầu video input)

**Lưu ý:** Bạn phải bật GPU trước khi chạy (Vào mục `Runtime` -> `Change runtime type` -> `Hardware accelerator` -> Chọn `T4 GPU`).

## 1. Mount Google Drive
Đầu tiên, hãy lưu video bạn muốn test lên Google Drive, sau đó chạy lệnh dưới để kết nối Drive với Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Repository và Cài đặt

In [ ]:
# TODO: Thay đổi link github của bạn vào đây. 
GIT_REPO = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git"
BRANCH = "feat/v3"

!rm -rf bradford_bulls
!git clone -b $BRANCH $GIT_REPO bradford_bulls

%cd /content/bradford_bulls/v3

# Xóa các package dev khỏi requirements để tránh xung đột với Colab
!sed -i '/ipykernel/d' requirements.txt
!sed -i '/jupyter/d' requirements.txt
!sed -i '/numpy/d' requirements.txt

!pip install -r requirements.txt

!mkdir -p weights
!wget -O weights/yolo11l.pt https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11l.pt

!python scripts/validate_setup.py

## 3. Restart Runtime (Quan trọng)
Nếu thông báo yêu cầu Restart session hiện ra ở ô trên, hãy ấn nút **Restart session**. Sau đó tiếp tục chạy các ô bên dưới.

## 4. Chuẩn bị Video Dữ Liệu
Bạn **chỉ cần duy nhất 1 video** để chạy pipeline lúc đầu. Các file metadata hay template ảnh logo là tùy chọn (chỉ cảnh báo chứ không lỗi).

In [ ]:
%cd /content/bradford_bulls/v3

!mkdir -p data/videos
!mkdir -p data/annotation_packages

# TODO: Đường dẫn video trên Google Drive của bạn
video_drive_path = "/content/drive/MyDrive/RugbyData/match_test.mp4"

!cp "$video_drive_path" data/videos/
!ls -la data/videos/

## 5. Chạy Pipeline
Chúng ta sẽ chạy Workflow 1 (dùng cờ `--kit-context` thay vì file meta) và giới hạn 60 giây để test nhanh.

In [ ]:
# Tên video (không có đuôi .mp4)
VIDEO_NAME = "match_test"

# Chạy với context là 'home' (hoặc 'away')
!python scripts/run_pipeline.py \
    --video data/videos/{VIDEO_NAME}.mp4 \
    --config configs/person_tracking.yaml \
    --output data/annotation_packages/{VIDEO_NAME} \
    --kit-context home \
    --max-duration 60

## 6. Lưu kết quả
Sau khi chạy xong, kết quả sẽ được nén và đẩy ngược về Drive của bạn.

In [ ]:
!zip -r {VIDEO_NAME}_package.zip data/annotation_packages/{VIDEO_NAME}
!cp {VIDEO_NAME}_package.zip /content/drive/MyDrive/RugbyData/
print("Hoàn tất!")